# A2.1 · "Who is calling?"

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A1.7 · Model routing architecture](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**.

| | |
|---|---|
| Open-source tooling | SPIRE |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Before agents, "who is calling?" had two well-understood answers.

**Human identity.** A person authenticates — password plus a second factor, or
SSO through an identity provider. The result is a token that says *this is
dana@corp*. It is short-lived, tied to a session, and revocable by disabling one
account. Crucially it carries an assumption: a human was present and intended
this.

**Workload identity.** A program authenticates. Historically this meant a
**service account** — a username and a long-lived secret, checked into a config
file and shared by everything in the deployment. Modern practice replaces the
secret with *attestation*: SPIFFE/SPIRE, IRSA, managed identities. The platform
vouches for the workload because of where it is running, so no secret has to be
planted anywhere. The result says *this is the payments-api pod in cluster-3*.

Both work because each answers a different question cleanly. A human token says
who *intended* something. A workload token says which *code* is running.

An agent breaks the distinction, because an agent is code that acts on a
human's intent, and the two identities have different lifetimes, different
scopes, and different revocation stories. Answer "who is calling?" with only
one of them and you lose information you will need later:

- Use the **human's** identity → the audit log says dana did it. She did not.
  She asked for something, three hops ago, and cannot tell you what happened.
- Use the **workload's** identity → the log says `triage-agent` did it, and now
  nobody knows *for whom*, or whether that person was allowed to ask.

The correct answer needs both at once, plus every intermediary. That is the rest
of this track.

## 2 · Demo — the two identities we already know how to do

Start with the familiar, working correctly. Nothing here is new; the point is to have both mechanisms concrete before we break them.

In [ ]:
import hashlib, json, time
from dataclasses import dataclass, field

# ---------- human identity: short-lived, session-bound, one account -------
@dataclass
class HumanToken:
    sub: str                       # dana@corp
    scopes: set
    auth_time: float = field(default_factory=time.time)
    amr: tuple = ("pwd", "mfa")    # how they proved it
    ttl: float = 3600

    @property
    def expired(self): return time.time() - self.auth_time > self.ttl
    def describe(self):
        return (f"human   sub={self.sub:14s} scopes={sorted(self.scopes)} "
                f"amr={list(self.amr)} ttl={self.ttl:.0f}s")

# ---------- workload identity: attested, no planted secret ---------------
@dataclass
class WorkloadToken:
    spiffe_id: str                 # spiffe://corp/ns/prod/sa/triage-agent
    scopes: set
    attested_by: str = "spire-agent on node-7"
    issued: float = field(default_factory=time.time)
    ttl: float = 300               # short, because it is cheap to reissue

    @property
    def expired(self): return time.time() - self.issued > self.ttl
    def describe(self):
        return (f"workload id={self.spiffe_id:38s} scopes={sorted(self.scopes)} "
                f"ttl={self.ttl:.0f}s")

dana = HumanToken("dana@corp", {"repo:read", "repo:write", "deploy:prod"})
svc  = WorkloadToken("spiffe://corp/ns/prod/sa/triage-agent", {"repo:read"})

print(dana.describe())
print(svc.describe())
print("\nBoth answer 'who is calling?' — for different questions:")
print("  human    → who INTENDED this")
print("  workload → which CODE is running")

## 3 · Where it breaks — the agent needs both, and gets one

Now put an agent in the middle. Dana asks the triage agent to fix a finding. The agent calls GitHub. What identity does GitHub see?

In practice, one of two things happens, and both lose information.

In [ ]:
def github_sees(token):
    """What the resource server can actually record."""
    if isinstance(token, HumanToken):
        return {"actor": token.sub, "on_behalf_of": token.sub,
                "audit_line": f"{token.sub} pushed a commit",
                "lost": "that an agent acted, and which one"}
    return {"actor": token.spiffe_id.split('/')[-1], "on_behalf_of": "unknown",
            "audit_line": f"{token.spiffe_id.split('/')[-1]} pushed a commit",
            "lost": "who asked for it, and whether they were allowed to"}

print("PATTERN 1 — hand the agent Dana's token ('it just needs her permissions')")
for k, v in github_sees(dana).items():
    print(f"   {k:14s} {v}")

print("\nPATTERN 2 — give the agent its own service account")
for k, v in github_sees(svc).items():
    print(f"   {k:14s} {v}")

print("\nBoth are complete, consistent records. Both are missing half the answer.")

## 4 · What the answer has to contain

Three facts, together, at the moment of the call:

- **`sub`** — the principal the action is *for* (Dana)
- **`actor`** — the identity actually making the call (`triage-agent`)
- **`act`** — the chain of everything in between

This is not invented for agents. It is [RFC 8693 token exchange](https://datatracker.ietf.org/doc/html/rfc8693), which OAuth has had since 2020 for exactly this problem, under the name **on-behalf-of**. A2.5 builds it properly. Here is just the shape, so the rest of the track has something to point at.

In [ ]:
@dataclass
class AgentToken:
    sub: str            # the human the action is for
    actor: str          # who is actually calling right now
    scopes: set
    act: dict = None    # nested chain of prior actors
    issued: float = field(default_factory=time.time)
    ttl: float = 300

    def chain(self):
        out, node = [], self.act
        while node:
            out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub:
            c.insert(0, self.sub)
        return c

    def describe(self):
        return (f"sub={self.sub}  actor={self.actor}\n"
                f"   chain  {' → '.join(self.chain())}\n"
                f"   scopes {sorted(self.scopes)}")

agent_tok = AgentToken(sub="dana@corp", actor="triage-agent",
                       scopes={"repo:read"}, act=None)
print(agent_tok.describe())

print("\nwhat GitHub can now record:")
print(f"   'triage-agent pushed a commit on behalf of dana@corp'")
print("   → both questions answered, one log line, nothing reconstructed later.")

## 5 · Verify — the three patterns side by side

In [ ]:
def audit(actor, on_behalf_of, chain):
    answerable = actor != "?" and on_behalf_of != "?" and chain != "not recorded"
    return {"who acted": actor, "for whom": on_behalf_of,
            "chain": chain, "answerable": answerable}

rows = {
 "agent uses Dana's token":  audit("dana@corp", "dana@corp", "not recorded"),
 "agent uses service acct":  audit("triage-agent", "?", "not recorded"),
 "on-behalf-of (RFC 8693)":  audit("triage-agent", "dana@corp",
                                   " → ".join(agent_tok.chain())),
}
for name, r in rows.items():
    print(f"{name:28s} answerable={str(r['answerable']):5s} "
          f"acted={r['who acted']:14s} for={r['for whom']:11s} {r['chain']}")
assert rows["on-behalf-of (RFC 8693)"]["answerable"]
print("\nOnly the third can answer an incident question without guesswork.")

## What you just proved

The human and workload tokens each print with their distinct properties. Both single-identity patterns produce a complete, consistent and incomplete audit record — one blaming Dana, one unable to say who asked. The on-behalf-of token records `dana@corp → triage-agent` and is the only one marked answerable.

## Your turn

Take one agent in your estate and find out which pattern it uses. The test is a single question: *for an action it took last week, can you name both the agent and the human who asked?* If you have to reconstruct it from timestamps, the answer is no.

---

**Next → [A2.2 · The bootstrap problem](https://spbreed.github.io/cyber-commons/lessons/A2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*